# Synthetic Data Generator
### Telecom Fault Ticket — Raw File Simulator

---

**Purpose:** Generate a realistic synthetic raw fault ticket file that mimics the structure,
distributions, and dirty-data patterns of the original dataset — so the full cleaning pipeline
(`pipeline.py`) can be run end-to-end without the real data.

**Design principles:**
- Matches the exact raw schema expected by `load_data()` (tab-delimited, same columns)
- Reproduces realistic dirty-data patterns: missing values, invalid priorities, bad RFO text,
  duplicate tickets, inconsistent formats — so the pipeline has real work to do
- Preserves realistic statistical distributions: Region 3 dominance (\~73% of volume),
  Priority 3 majority (\~87%), SLA compliance \~88%, MTTR \~61h
- Generates a synthetic site database (`site_database_synthetic.csv`) with matching PLAID codes
  so Region 3 zone assignment works correctly
- **No real data is used or reverse-engineered** — all values are statistically plausible

## 1. Setup & Configuration

In [1]:
import os
import sys
import numpy as np
import pandas as pd

os.chdir(os.path.join('..', '..'))
if os.path.abspath(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.path.abspath(os.getcwd()))

rng = np.random.default_rng(42)

CLEAN_OUT      = 'data/raw/fault_tickets/clean_synthetic_dataset.csv'
SITE_DB_OUT  = 'data/external/site_database_synthetic.csv'

os.makedirs('data/raw/fault_tickets', exist_ok=True)
os.makedirs('data/external',           exist_ok=True)

print("✅ Section 1 complete — paths and config ready")

✅ Section 1 complete — paths and config ready


## 2. Reference Tables

In [2]:
# ── Snapshot targets (section 1a + 2b) ───────────────────────────────────────
SNAPSHOT_TARGETS = {
    'ZONE 1': dict(tickets=4306, sla=83.1, mttr=62.6, fd=64.6),
    'ZONE 2': dict(tickets=7307, sla=83.9, mttr=51.0, fd=62.8),
    'ZONE 3': dict(tickets=4393, sla=80.6, mttr=32.3, fd=86.1),
    'ZONE 4': dict(tickets=7952, sla=84.5, mttr=47.4, fd=56.2),
    'ZONE 5': dict(tickets=9131, sla=79.6, mttr=77.8, fd=53.2),
    'ZONE 6': dict(tickets=3818, sla=80.9, mttr=70.0, fd=61.4),
}

# Resolution path % per zone — snapshot section 2b
# Auto%, NOC%, FD% must sum to 100%.  FD = 100 - Auto - NOC.
AUTO_PCT = {'ZONE 1':  1.3, 'ZONE 2':  1.6, 'ZONE 3':  0.5,
            'ZONE 4': 11.4, 'ZONE 5': 12.8, 'ZONE 6':  1.4}
NOC_PCT  = {'ZONE 1': 34.1, 'ZONE 2': 35.6, 'ZONE 3': 13.4,
            'ZONE 4': 32.4, 'ZONE 5': 34.0, 'ZONE 6': 37.2}

# ── Priority × Urgency ticket counts per zone ─────────────────────────────────
# (n_breach, n_compliant) — sum must equal SNAPSHOT_TARGETS[zone]['tickets']
PU_SLA_TABLE = {
    'ZONE 1': {'1.1': (5,6),  '1.3': (0,2),  '2.1': (5,4),   '2.2': (2,7),   '2.3': (66,193),
               '3.1': (226,1445), '3.2': (136,222), '3.3': (289,1698)},
    'ZONE 2': {'1.1': (8,19), '2.1': (10,50), '2.2': (3,27),  '2.3': (154,280),
               '3.1': (333,2391), '3.2': (349,754), '3.3': (322,2607)},
    'ZONE 3': {'1.1': (2,2),  '2.1': (0,6),  '2.2': (0,13),  '2.3': (37,49),
               '3.1': (257,1037), '3.2': (237,321), '3.3': (320,2112)},
    'ZONE 4': {'1.1': (4,19), '1.3': (0,2),  '2.1': (12,27), '2.2': (5,50),  '2.3': (131,1032),
               '3.1': (252,2232), '3.2': (395,753), '3.3': (430,2608)},
    'ZONE 5': {'1.1': (6,32), '2.1': (9,33), '2.2': (141,611),'2.3': (142,456),
               '3.1': (346,2524), '3.2': (607,955), '3.3': (613,2656)},
    'ZONE 6': {'1.1': (10,32),'2.1': (0,8),  '2.2': (1,22),  '2.3': (25,82),
               '3.1': (210,1344), '3.2': (202,255), '3.3': (283,1344)},
}

# ── RFO counts per zone (snapshot section 2d + zone breakdown) ────────────────
ZONE_RFO_COUNTS = {
    'ZONE 1': {
        'ADMIN-Lessor Related Cause': 87, 'EQUIPMENT-Configuration Problem': 85,
        'EQUIPMENT-Decommissioned': 1, 'EQUIPMENT-Defective Hardware': 887,
        'EQUIPMENT-Software Problem': 2, 'FACILITIES-Cooling System Problem': 319,
        'FACILITIES-Design Problem': 1, 'FACILITIES-Monitoring Equipment Problem': 2,
        'FACILITIES-Power Failure': 1271, 'FOC CUT - LINEAR': 540,
        'FOC CUT WITH REDUNDANCY': 84, 'OTHERS-Preventive Maintenance': 30,
        'OTHERS-Security Breach': 6, 'THIRD PARTY-Activity Related': 20,
        'THIRD PARTY-POI': 1, 'TRANSMISSION-Cable Problem': 40,
        'TRANSMISSION-Equipment Problem': 14, 'TRANSMISSION-Fiber Problem': 6,
        'TRANSMISSION-IP Network Problem': 243,
        'TRANSMISSION-Microwave Antenna System Problem': 16,
        'UNKNOWN-Under Investigation': 651,
    },
    'ZONE 2': {
        'ADMIN-Lessor Related Cause': 75, 'EQUIPMENT-Configuration Problem': 127,
        'EQUIPMENT-Decommissioned': 5, 'EQUIPMENT-Defective Hardware': 1245,
        'EQUIPMENT-Software Problem': 10, 'FACILITIES-Cooling System Problem': 291,
        'FACILITIES-Design Problem': 1, 'FACILITIES-Monitoring Equipment Problem': 4,
        'FACILITIES-Power Failure': 2731, 'FACILITIES-Water Seepage': 4,
        'FOC CUT - LINEAR': 1345, 'FOC CUT WITH REDUNDANCY': 144,
        'OTHERS-Force Majeure': 2, 'OTHERS-Preventive Maintenance': 8,
        'OTHERS-Security Breach': 9, 'THIRD PARTY-Activity Related': 55,
        'THIRD PARTY-POI': 90, 'TRANSMISSION-Cable Problem': 64,
        'TRANSMISSION-Equipment Problem': 26, 'TRANSMISSION-Fiber Problem': 61,
        'TRANSMISSION-IP Network Problem': 301,
        'TRANSMISSION-Microwave Antenna System Problem': 14,
        'TRANSMISSION-Offnetwork': 36, 'UNKNOWN-Under Investigation': 659,
    },
    'ZONE 3': {
        'ADMIN-Lessor Related Cause': 80, 'EQUIPMENT-Configuration Problem': 154,
        'EQUIPMENT-Decommissioned': 1, 'EQUIPMENT-Defective Hardware': 1103,
        'EQUIPMENT-Software Problem': 1, 'FACILITIES-Cooling System Problem': 313,
        'FACILITIES-Design Problem': 2, 'FACILITIES-Monitoring Equipment Problem': 13,
        'FACILITIES-Power Failure': 1647, 'FACILITIES-Water Seepage': 7,
        'FOC CUT - LINEAR': 514, 'FOC CUT WITH REDUNDANCY': 81,
        'OTHERS-Force Majeure': 2, 'OTHERS-Preventive Maintenance': 4,
        'OTHERS-Security Breach': 12, 'THIRD PARTY-Activity Related': 32,
        'THIRD PARTY-POI': 1, 'TRANSMISSION-Cable Problem': 48,
        'TRANSMISSION-Capacity Problem': 1, 'TRANSMISSION-Equipment Problem': 10,
        'TRANSMISSION-Fiber Problem': 9, 'TRANSMISSION-IP Network Problem': 99,
        'TRANSMISSION-Microwave Antenna System Problem': 40,
        'UNKNOWN-Under Investigation': 219,
    },
    'ZONE 4': {
        'ADMIN-Lessor Related Cause': 115, 'EQUIPMENT-Configuration Problem': 194,
        'EQUIPMENT-Decommissioned': 3, 'EQUIPMENT-Defective Hardware': 1792,
        'EQUIPMENT-Software Problem': 7, 'FACILITIES-Cooling System Problem': 387,
        'FACILITIES-Design Problem': 11, 'FACILITIES-Monitoring Equipment Problem': 2,
        'FACILITIES-Power Failure': 2736, 'FACILITIES-Water Seepage': 6,
        'FOC CUT - LINEAR': 556, 'FOC CUT WITH REDUNDANCY': 109,
        'OTHERS-Force Majeure': 2, 'OTHERS-Preventive Maintenance': 11,
        'OTHERS-Security Breach': 8, 'THIRD PARTY-Activity Related': 81,
        'THIRD PARTY-POI': 35, 'TRANSMISSION-Cable Problem': 65,
        'TRANSMISSION-Equipment Problem': 29, 'TRANSMISSION-Fiber Problem': 24,
        'TRANSMISSION-IP Network Problem': 285,
        'TRANSMISSION-Microwave Antenna System Problem': 11,
        'UNKNOWN-Under Investigation': 1483,
    },
    'ZONE 5': {
        'ADMIN-Lessor Related Cause': 82, 'EQUIPMENT-Configuration Problem': 237,
        'EQUIPMENT-Decommissioned': 2, 'EQUIPMENT-Defective Hardware': 2070,
        'EQUIPMENT-Software Problem': 9, 'FACILITIES-Cooling System Problem': 587,
        'FACILITIES-Design Problem': 1, 'FACILITIES-Monitoring Equipment Problem': 4,
        'FACILITIES-Power Failure': 3251, 'FACILITIES-Water Seepage': 8,
        'FOC CUT - LINEAR': 381, 'FOC CUT WITH REDUNDANCY': 65,
        'OTHERS-Preventive Maintenance': 26, 'OTHERS-Security Breach': 13,
        'THIRD PARTY-Activity Related': 101, 'THIRD PARTY-POI': 120,
        'TRANSMISSION-Cable Problem': 84, 'TRANSMISSION-Capacity Problem': 3,
        'TRANSMISSION-Equipment Problem': 23, 'TRANSMISSION-Fiber Problem': 14,
        'TRANSMISSION-IP Network Problem': 259,
        'TRANSMISSION-Microwave Antenna System Problem': 71,
        'UNKNOWN-Under Investigation': 1720,
    },
    'ZONE 6': {
        'ADMIN-Lessor Related Cause': 108, 'EQUIPMENT-Configuration Problem': 83,
        'EQUIPMENT-Defective Hardware': 824, 'EQUIPMENT-Software Problem': 2,
        'FACILITIES-Cooling System Problem': 239, 'FACILITIES-Design Problem': 6,
        'FACILITIES-Monitoring Equipment Problem': 7, 'FACILITIES-Power Failure': 1419,
        'FACILITIES-Water Seepage': 5, 'FOC CUT - LINEAR': 399,
        'FOC CUT WITH REDUNDANCY': 40, 'OTHERS-Force Majeure': 2,
        'OTHERS-Preventive Maintenance': 11, 'OTHERS-Security Breach': 4,
        'THIRD PARTY-Activity Related': 48, 'TRANSMISSION-Cable Problem': 56,
        'TRANSMISSION-Equipment Problem': 6, 'TRANSMISSION-Fiber Problem': 8,
        'TRANSMISSION-IP Network Problem': 106,
        'TRANSMISSION-Microwave Antenna System Problem': 48,
        'UNKNOWN-Under Investigation': 397,
    },
}

# ── RFO → Fault Type ──────────────────────────────────────────────────────────
RFO_TO_FAULT_TYPE = {
    'EQUIPMENT-Defective Hardware': 'EQUIPMENT',
    'EQUIPMENT-Configuration Problem': 'EQUIPMENT',
    'EQUIPMENT-Software Problem': 'EQUIPMENT',
    'EQUIPMENT-Decommissioned': 'EQUIPMENT',
    'FACILITIES-Power Failure': 'FACILITIES',
    'FACILITIES-Cooling System Problem': 'FACILITIES',
    'FACILITIES-Monitoring Equipment Problem': 'FACILITIES',
    'FACILITIES-Water Seepage': 'FACILITIES',
    'FACILITIES-Design Problem': 'FACILITIES',
    'TRANSMISSION-IP Network Problem': 'TRANSMISSION',
    'TRANSMISSION-Cable Problem': 'TRANSMISSION',
    'TRANSMISSION-Microwave Antenna System Problem': 'TRANSMISSION',
    'TRANSMISSION-Fiber Problem': 'TRANSMISSION',
    'TRANSMISSION-Equipment Problem': 'TRANSMISSION',
    'TRANSMISSION-Offnetwork': 'TRANSMISSION',
    'TRANSMISSION-Capacity Problem': 'TRANSMISSION',
    'FOC CUT - LINEAR': 'OTHERS',
    'FOC CUT WITH REDUNDANCY': 'OTHERS',
    'ADMIN-Lessor Related Cause': 'OTHERS',
    'THIRD PARTY-Activity Related': 'OTHERS',
    'THIRD PARTY-POI': 'OTHERS',
    'OTHERS-Preventive Maintenance': 'OTHERS',
    'OTHERS-Security Breach': 'OTHERS',
    'OTHERS-Force Majeure': 'OTHERS',
    'UNKNOWN-Under Investigation': 'UNKNOWN',
}

# ── Zone → Cities (city name, ticket allocation) ─────────────────────────────
ZONE_CITIES = {
    'ZONE 1': [('CALOOCAN CITY', 2282), ('VALENZUELA CITY', 1845), ('NAVOTAS', 179)],
    'ZONE 2': [('QUEZON CITY', 7307)],
    'ZONE 3': [('RIZAL', 2723), ('MALABON CITY', 974), ('MARIKINA CITY', 696)],
    'ZONE 4': [('MANILA', 3589), ('PASIG CITY', 1791), ('SAN JUAN CITY', 1451),
               ('MANDALUYONG', 1121)],
    'ZONE 5': [('MAKATI CITY', 5145), ('TAGUIG CITY', 2342), ('PASAY CITY', 1601),
               ('PATEROS', 43)],
    'ZONE 6': [('PARANAQUE CITY', 1775), ('MUNTINLUPA CITY', 1030),
               ('LAS PINAS CITY', 996), ('BACOOR', 17)],
}

print("✅ Section 2 complete — reference tables loaded")

✅ Section 2 complete — reference tables loaded


## 3. Generate Site Database

In [3]:
def generate_site_db(zone_cities, sites_per_city=35):
    records = []
    for zone, cities in zone_cities.items():
        z_abbrev = zone.replace('ZONE ', 'Z')
        for city, _ in cities:
            c_abbrev = city[:3].replace(' ', '').upper()
            for seq in range(1, sites_per_city + 1):
                records.append({
                    'PLAID':      f'{z_abbrev}{c_abbrev}{seq:04d}',
                    'AssignArea': zone,
                    'AssignCity': city,
                })
    df = pd.DataFrame(records)
    df.to_csv(SITE_DB_OUT, index=False)
    print(f"✅ Section 3 complete — synthetic site database generated")
    print(f"   Rows   : {len(df):,}")
    print(f"   Saved  : {SITE_DB_OUT}")
    return df

df_site = generate_site_db(ZONE_CITIES)

def get_plaid(city):
    pool = df_site[df_site['AssignCity'] == city]['PLAID'].values
    return rng.choice(pool) if len(pool) > 0 else 'UNKNOWN_PLAID'

✅ Section 3 complete — synthetic site database generated
   Rows   : 665
   Saved  : data/external/site_database_synthetic.csv


## 4. Generate Raw Tickets

### Generate Clean Tickets

In [4]:
def generate_tickets():
    """
    Build the synthetic ticket DataFrame with exact KPI targets.

    Resolution path assignment (Fix 2 + Fix 3):
      Each zone's n_auto, n_noc, n_fd are computed from section-2b percentages.
      Tickets are assigned to paths independently of SLA_Compliant, then shuffled.
        - Auto  → FT_Owner=NaN, WOLeadName=NaN   (pipeline → Auto_Self_Restored)
        - NOC   → FT_Owner='NOC_TEAM', WOLeadName=NaN (pipeline → NOC_Remote_Restored)
        - FD    → FT_Owner='FIELD_TEAM', WOLeadName='FIELD_LEAD' (pipeline → Field_Dispatch_Restored)

    OUTAGEDURATION (Fix 1):
      Draw from Normal(mean=target_mttr, sd=0.3*mttr), clip to >0, then
      mean-shift the entire zone vector so mean==target exactly.
    """
    all_rows = []
    ticket_id = 1

    for zone, pu_dict in PU_SLA_TABLE.items():
        tgt      = SNAPSHOT_TARGETS[zone]
        n_total  = tgt['tickets']
        mttr     = tgt['mttr']
        sla_pct  = tgt['sla'] / 100

        # ── Step A: build flat lists of (priority, urgency, sla_compliant) ──
        pu_labels = []   # (priority, urgency, sla_compliant)
        for pu, (n_bad, n_good) in pu_dict.items():
            p, u = map(int, pu.split('.'))
            pu_labels.extend([(p, u, 1)] * n_good)
            pu_labels.extend([(p, u, 0)] * n_bad)

        assert len(pu_labels) == n_total, \
            f"{zone}: PU table has {len(pu_labels)} rows, expected {n_total}"

        # ── Step B: resolution path counts (Fix 2 + Fix 3) ─────────────────
        n_auto = round(n_total * AUTO_PCT[zone] / 100)
        n_noc  = round(n_total * NOC_PCT[zone]  / 100)
        n_fd   = n_total - n_auto - n_noc

        # Build path label array and shuffle so path is independent of SLA
        path_labels = (
            ['Auto']  * n_auto +
            ['NOC']   * n_noc  +
            ['FD']    * n_fd
        )
        rng.shuffle(path_labels)

        # ── Step C: OUTAGEDURATION — mean-preserved Normal (Fix 1) ─────────
        spread    = mttr * 0.30
        durations = rng.normal(loc=mttr, scale=spread, size=n_total)
        durations = np.clip(durations, 0.5, None)
        durations = durations - durations.mean() + mttr   # exact mean = target

        # ── Step D: RFO pool for this zone ──────────────────────────────────
        rfo_pool = []
        for rfo, cnt in ZONE_RFO_COUNTS[zone].items():
            rfo_pool.extend([rfo] * cnt)
        assert len(rfo_pool) == n_total, \
            f"{zone}: RFO pool has {len(rfo_pool)} rows, expected {n_total}"
        rng.shuffle(rfo_pool)

        # ── Step E: city weights for random city assignment ──────────────────
        cities      = [c for c, _ in ZONE_CITIES[zone]]
        city_counts = [cnt for _, cnt in ZONE_CITIES[zone]]
        city_probs  = np.array(city_counts, dtype=float)
        city_probs /= city_probs.sum()

        # ── Step F: emit one row per ticket ─────────────────────────────────
        for i in range(n_total):
            p, u, sla_ok = pu_labels[i]
            path         = path_labels[i]
            rfo          = rfo_pool[i]
            dur          = float(durations[i])
            city         = rng.choice(cities, p=city_probs)
            plaid        = get_plaid(city)

            # Resolution-path columns understood by pipeline Step 22
            if path == 'Auto':
                ft_owner    = np.nan
                wo_lead     = np.nan
                wo_group    = np.nan
            elif path == 'NOC':
                ft_owner    = 'NOC_TEAM'
                wo_lead     = np.nan
                wo_group    = 'NOC-RAN'
            else:  # FD
                ft_owner    = 'FIELD_TEAM'
                wo_lead     = 'FIELD_LEAD'
                wo_group    = 'FO_GMA'

            # NOC time: Auto/NOC have no physical dispatch (near-zero);
            # FD gets a small positive dispatch delay
            dispatch_delay = 0.0 if path in ('Auto', 'NOC') else rng.uniform(0.1, 1.5)
            field_time     = dur - dispatch_delay if path == 'FD' else 0.0

            all_rows.append({
                'TICKETID':               f'TKT{ticket_id:08d}',
                'Region':                 'Region 3',
                'ZONE':                   zone,
                'CITY':                   city,
                'Area':                   plaid,
                'FT Status':              'RESOLVED',
                'Priority':               p,
                'Urgency':                u,
                'OUTAGEDURATION':         round(dur, 4),
                'RESOLUTION_TIME_HOURS':  round(dur, 4),
                'Standardized RFO':       rfo,
                'RFODescription':         rfo,
                'RootCause':              f'Resolved – {rfo}',
                'Fault Type':             RFO_TO_FAULT_TYPE.get(rfo, 'UNKNOWN'),
                'StartDateTime':          '2023-03-15 09:00:00',
                'ActionTaken':            'Service restored',
                'NEType':                 rng.choice(
                                              ['BTS', 'NODEB', 'SDH_MUX', 'ROUTER'],
                                              p=[0.70, 0.15, 0.10, 0.05]),
                'SLA_Compliant':          sla_ok,
                'FT_Owner':               ft_owner,
                'WOLeadName':             wo_lead,
                'WOOwnerGroup':           wo_group,
                'FIELD_TIME_HOURS':       round(field_time, 4),
                'DISPATCH_DELAY_HOURS':   round(dispatch_delay, 4),
            })
            ticket_id += 1

    return pd.DataFrame(all_rows)


# ── Build dataset ─────────────────────────────────────────────────────────────
df_clean = generate_tickets()
print(f"Generated tickets: {len(df_clean):,} rows")

# Final columns
df_clean['Priority_Urgency'] = (
    df_clean['Priority'].astype(str) + '.' + df_clean['Urgency'].astype(str)
)
df_clean['REPORTDATE'] = (
    pd.to_datetime(df_clean['StartDateTime'])
    - pd.to_timedelta(rng.uniform(0, 4, len(df_clean)), unit='h')
).dt.strftime('%Y-%m-%d %H:%M:%S')
df_clean['ContactGroup'] = 'GMA_CORE'
df_clean['DESCRIPTION']  = (
    'Fault in ' + df_clean['CITY'] +
    ' (' + df_clean['ZONE'] + ') – ' + df_clean['RFODescription']
)
df_clean['TICKETID'] = [f'TKT{i:08d}' for i in range(1, len(df_clean) + 1)]

# ── Pre-save self-audit ───────────────────────────────────────────────────────
print("\n── Pre-save audit ───────────────────────────────────────────────────────")
for zone, tgt in SNAPSHOT_TARGETS.items():
    z = df_clean[df_clean['ZONE'] == zone]
    n        = len(z)
    sla_pct  = z['SLA_Compliant'].mean() * 100
    mttr_val = z['OUTAGEDURATION'].mean()

    # Resolution path
    auto_n = ((z['FT_Owner'].isna()) & (z['WOLeadName'].isna())).sum()
    noc_n  = (z['FT_Owner'].notna() & z['WOLeadName'].isna()).sum()
    fd_n   = (z['FT_Owner'].notna() & z['WOLeadName'].notna()).sum()
    fd_pct = fd_n / n * 100

    t_ok  = '✅' if n       == tgt['tickets'] else f"❌ got {n}"
    s_ok  = '✅' if abs(sla_pct  - tgt['sla'])  < 0.05 else f"❌ got {sla_pct:.2f}%"
    m_ok  = '✅' if abs(mttr_val - tgt['mttr']) < 0.01  else f"❌ got {mttr_val:.4f}h"
    f_ok  = '✅' if abs(fd_pct   - tgt['fd'])   < 0.20  else f"❌ got {fd_pct:.1f}%"

    print(f"  {zone}  tickets={n} {t_ok}  SLA={sla_pct:.1f}% {s_ok}  "
          f"MTTR={mttr_val:.2f}h {m_ok}  FD={fd_pct:.1f}% {f_ok}  "
          f"(Auto={auto_n} NOC={noc_n} FD={fd_n})")

total_auto = (df_clean['FT_Owner'].isna() & df_clean['WOLeadName'].isna()).sum()
total_noc  = (df_clean['FT_Owner'].notna() & df_clean['WOLeadName'].isna()).sum()
total_fd   = (df_clean['FT_Owner'].notna() & df_clean['WOLeadName'].notna()).sum()
print(f"\n  NCR totals → Auto={total_auto:,} ({total_auto/len(df_clean)*100:.1f}%)  "
      f"NOC={total_noc:,} ({total_noc/len(df_clean)*100:.1f}%)  "
      f"FD={total_fd:,} ({total_fd/len(df_clean)*100:.1f}%)")
print(f"  Expected  → Auto=2,325 (6.3%)  NOC=11,760 (31.9%)  FD=22,822 (61.8%)")

print("✅ 36,907 rows | exact tickets, SLA%, MTTR, FD% per zone | "
      "Auto/NOC/FD paths decoupled from SLA")

Generated tickets: 36,907 rows

── Pre-save audit ───────────────────────────────────────────────────────
  ZONE 1  tickets=4306 ✅  SLA=83.1% ✅  MTTR=62.60h ✅  FD=64.6% ✅  (Auto=56 NOC=1468 FD=2782)
  ZONE 2  tickets=7307 ✅  SLA=83.9% ✅  MTTR=51.00h ✅  FD=62.8% ✅  (Auto=117 NOC=2601 FD=4589)
  ZONE 3  tickets=4393 ✅  SLA=80.6% ✅  MTTR=32.30h ✅  FD=86.1% ✅  (Auto=22 NOC=589 FD=3782)
  ZONE 4  tickets=7952 ✅  SLA=84.5% ✅  MTTR=47.40h ✅  FD=56.2% ✅  (Auto=907 NOC=2576 FD=4469)
  ZONE 5  tickets=9131 ✅  SLA=79.6% ✅  MTTR=77.80h ✅  FD=53.2% ✅  (Auto=1169 NOC=3105 FD=4857)
  ZONE 6  tickets=3818 ✅  SLA=80.9% ✅  MTTR=70.00h ✅  FD=61.4% ✅  (Auto=53 NOC=1420 FD=2345)

  NCR totals → Auto=2,324 (6.3%)  NOC=11,759 (31.9%)  FD=22,824 (61.8%)
  Expected  → Auto=2,325 (6.3%)  NOC=11,760 (31.9%)  FD=22,822 (61.8%)
✅ 36,907 rows | exact tickets, SLA%, MTTR, FD% per zone | Auto/NOC/FD paths decoupled from SLA


### Inject Dirty Data

In [5]:
from config import EXPECTED_RFO_VALUES

rng = np.random.default_rng(99)

RAW_OUT = 'data/raw/fault_tickets/synthetic_dataset.csv'

# ── Load clean base ───────────────────────────────────────────────────────────
df = df_clean
print(f"Loaded clean base: {len(df):,} rows")
assert len(df) == 36_907, f"Expected 36,907 rows, got {len(df)}"

n_sj_actual  = int((df['CITY'] == 'SAN JUAN CITY').sum())
N_SJ_UNKNOWN    = round(n_sj_actual * 0.637)          # 63.7% → 912  (UNKNOWN RFO)
N_SJ_ROC        = round(n_sj_actual * 0.101)          # 10.1% → 145  (ROC account)
TARGET_UNK_EID  = round(n_sj_actual * 0.576)          # 57.6% → 825  (no WOLeadName POST-ROC)
N_SJ_NO_LEAD    = TARGET_UNK_EID + N_SJ_ROC           #         → 970  blank before ROC restore

def _sample_excl_sj(n):
    pool = df[df['CITY'] != 'SAN JUAN CITY']
    return pool.iloc[rng.choice(len(pool), size=n, replace=True)].copy().reset_index(drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# PART A — DROPPABLE ROWS (1,412 total)
# ─────────────────────────────────────────────────────────────────────────────

dirty_chunks = []
next_id = 37_000

def _new_ids(n):
    global next_id
    ids = [f'TKT{i:08d}' for i in range(next_id, next_id + n)]
    next_id += n
    return ids

# A1: Priority 4 (700) — dropped at Step 3
a1 = _sample_excl_sj(700)
a1['TICKETID']       = _new_ids(700)
a1['Priority']       = 4
a1['Urgency']        = rng.integers(1, 4, size=700)
a1['FT Status']      = rng.choice(['CLOSED', 'RESOLVED'], size=700)
a1['Fault Type']     = rng.choice(['EQUIPMENT', 'FACILITIES', 'TRANSMISSION', 'OTHERS'], size=700)
a1['OUTAGEDURATION'] = rng.uniform(48, 200, size=700).round(2)
dirty_chunks.append(a1)

# A2: CANCELLED status (280) — dropped at Step 5
a2 = _sample_excl_sj(280)
a2['TICKETID']       = _new_ids(280)
a2['FT Status']      = 'CANCELLED'
a2['OUTAGEDURATION'] = rng.uniform(0.1, 5, size=280).round(2)
dirty_chunks.append(a2)

# A3: INPROG status (150) — dropped at Step 5
a3 = _sample_excl_sj(150)
a3['TICKETID']       = _new_ids(150)
a3['FT Status']      = 'INPROG'
a3['OUTAGEDURATION'] = np.nan
dirty_chunks.append(a3)

# A4: INVALID Fault Type (100) — dropped at Step 6
a4 = _sample_excl_sj(100)
a4['TICKETID']   = _new_ids(100)
a4['Fault Type'] = 'INVALID'
dirty_chunks.append(a4)

# A5: Blank Priority (80) — dropped at Step 2
a5 = _sample_excl_sj(80)
a5['TICKETID'] = _new_ids(80)
a5['Priority'] = np.nan
dirty_chunks.append(a5)

# A6: Duplicate TICKETIDs (62) — pipeline drops second occurrence at Step 1
a6 = df[df['CITY'] != 'SAN JUAN CITY'].sample(n=62, random_state=7).copy()
a6['OUTAGEDURATION'] = a6['OUTAGEDURATION'] * rng.uniform(0.9, 1.1, size=62)
dirty_chunks.append(a6)

# A7: Bad Urgency (25) — dropped at Step 4
a7 = _sample_excl_sj(25)
a7['TICKETID'] = _new_ids(25)
a7['Urgency']  = rng.choice([9, 99, -1], size=25)
dirty_chunks.append(a7)

# A8: Uncategorizable RFO (15) — dropped at Steps 13/15
GIBBERISH_RFOS = [
    'XYZ-UNKNOWN-CODE', 'MISC-ADMIN-PENDING', 'LEGACY-CODE-999',
    'TEMP-PLACEHOLDER', 'ERROR-NO-RFO', 'AWAITING-CLASSIFICATION',
    'OBSOLETE-CATEGORY', 'IMPORT-ERROR', 'NULL-CAUSE', 'TBD-REVIEW',
    'DEPRECATED-TAG', 'SYSTEM-GENERATED', 'BATCH-IMPORT', 'PLACEHOLDER-X',
    'UNKNOWN-LEGACY',
]
a8 = _sample_excl_sj(15)
a8['TICKETID']       = _new_ids(15)
a8['RFODescription'] = GIBBERISH_RFOS
a8['RootCause']      = GIBBERISH_RFOS
a8['ActionTaken']    = 'Pending review'
dirty_chunks.append(a8)

total_dirty = sum(len(c) for c in dirty_chunks)
assert total_dirty == 1412, f"Expected 1412, got {total_dirty}"
print(f"  Part A — droppable rows: {total_dirty} ✅")

# ─────────────────────────────────────────────────────────────────────────────
# PART B — TIMESTAMP CORRUPTION (1,327 rows; stays in dataset)
# ─────────────────────────────────────────────────────────────────────────────

ZONE_TICKETS = {
    'ZONE 1': 4306, 'ZONE 2': 7307, 'ZONE 3': 4393,
    'ZONE 4': 7952, 'ZONE 5': 9131, 'ZONE 6': 3818,
}
other_corrupt = 1134
other_total   = sum(v for k, v in ZONE_TICKETS.items() if k != 'ZONE 1')
zone_corrupt  = {'ZONE 1': 193}
remaining     = other_corrupt
other_zones   = [(z, c) for z, c in ZONE_TICKETS.items() if z != 'ZONE 1']
for i, (zone, n) in enumerate(other_zones):
    cnt = remaining if i == len(other_zones) - 1 else round(other_corrupt * n / other_total)
    zone_corrupt[zone] = cnt
    remaining -= cnt

df_work = df.copy()
for zone, n_corrupt in zone_corrupt.items():
    zone_idx = df_work[df_work['ZONE'] == zone].index.tolist()
    df_work.loc[rng.choice(zone_idx, size=n_corrupt, replace=False),
                'StartDateTime'] = '2023-03-17 09:00:00'

print(f"  Part B — timestamp corruption: {sum(zone_corrupt.values())} rows "
      f"(Zone 1: {zone_corrupt['ZONE 1']}) ✅")

# ─────────────────────────────────────────────────────────────────────────────
# PART C — RFO NOISE (15% of non-SJ rows)
# ─────────────────────────────────────────────────────────────────────────────

RFO_NOISE_MAP = {
    'FACILITIES-Power Failure': [
        'power outage', 'ac mains failure', 'commercial power loss',
        'genset failure', 'dc under voltage alarm', 'facilities power',
        'power supply issue', 'rectifier fault',
    ],
    'EQUIPMENT-Defective Hardware': [
        'defective card', 'rru issue', 'hardware failure', 'module failure',
        'antenna fault', 'equipment failure', 'card module replaced', 'bts antenna problem',
    ],
    'FOC CUT - LINEAR': [
        'single fiber cut', 'foc cut linear', 'fiber break linear',
        'transmission foc cut', 'fiber optic cut',
    ],
    'FOC CUT WITH REDUNDANCY': [
        'multiple fiber cut', 'foc cut redundancy', 'redundant link failure',
        'foc cut with redundancy', 'multiple access link failure',
    ],
    'FACILITIES-Cooling System Problem': [
        'aircon failure', 'cooling system issue', 'temperature problem', 'ac unit fault',
    ],
    'TRANSMISSION-IP Network Problem': [
        'ip network issue', 'link down', 'logical trail', 'connectivity issue',
    ],
    'UNKNOWN-Under Investigation': [
        'under investigation', 'bau open', 'tba', 'inquiry', 'unknown',
    ],
    'EQUIPMENT-Configuration Problem': [
        'configuration error', 'misconfiguration', 'config issue', 'reconfigured',
    ],
    'ADMIN-Lessor Related Cause': [
        'site access restriction', 'lessor issue', 'extortion', 'access denied',
    ],
    'TRANSMISSION-Cable Problem': [
        'transmission cable', 'cable damage', 'cable issue',
    ],
}

non_sj_idx   = df_work[df_work['CITY'] != 'SAN JUAN CITY'].index.tolist()
noise_chosen = rng.choice(non_sj_idx, size=int(len(non_sj_idx) * 0.15), replace=False)
for idx in noise_chosen:
    canonical = df_work.at[idx, 'Standardized RFO']
    if canonical in RFO_NOISE_MAP:
        df_work.at[idx, 'RFODescription'] = rng.choice(RFO_NOISE_MAP[canonical])

print(f"  Part C — RFO noise: {len(noise_chosen):,} rows ({len(noise_chosen)/len(non_sj_idx)*100:.0f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# PART D — MISSING NEType (629 non-SJ rows)
# ─────────────────────────────────────────────────────────────────────────────

df_work.loc[rng.choice(non_sj_idx, size=629, replace=False), 'NEType'] = np.nan
print(f"  Part D — missing NEType: 629 rows")

# ─────────────────────────────────────────────────────────────────────────────
# PART E — SAN JUAN CITY ANOMALY (exact-count, zero-baseline approach)
#
# Zero-baseline = force ALL SJ rows to non-UNKNOWN before setting chosen rows,
# so the count is EXACTLY N_SJ_UNKNOWN regardless of how many were UNKNOWN before.
# ─────────────────────────────────────────────────────────────────────────────

FALLBACK_RFO = 'FACILITIES-Power Failure'   # used where clean value is UNKNOWN
sj_idx       = df_work[df_work['CITY'] == 'SAN JUAN CITY'].index.tolist()
sj_clean_rfo = df.loc[sj_idx, 'RFODescription'].values

# Step 1: zero-baseline — no SJ row is UNKNOWN
non_unk_rfo = np.where(sj_clean_rfo == 'UNKNOWN-Under Investigation', FALLBACK_RFO, sj_clean_rfo)
df_work.loc[sj_idx, 'RFODescription']   = non_unk_rfo
df_work.loc[sj_idx, 'Standardized RFO'] = non_unk_rfo

# Step 2: set EXACTLY N_SJ_UNKNOWN rows to UNKNOWN
sj_unk_chosen = rng.choice(sj_idx, size=N_SJ_UNKNOWN, replace=False)
df_work.loc[sj_unk_chosen, 'RFODescription']   = 'UNKNOWN-Under Investigation'
df_work.loc[sj_unk_chosen, 'Standardized RFO'] = 'UNKNOWN-Under Investigation'
df_work.loc[sj_unk_chosen, 'RootCause']        = 'Under investigation'

# Step 3: WOLeadName zero-baseline — fill ALL SJ to non-NaN, then blank exactly N_SJ_NO_LEAD
# (Clean base has 640 NaN from NOC/Auto paths. Zero-baseline prevents leakage.)
df_work.loc[sj_idx, 'WOLeadName'] = 'FIELD_LEAD'
df_work.loc[sj_idx, 'FT_Owner']   = 'FIELD_TEAM'

# Step 4: blank EXACTLY N_SJ_NO_LEAD rows (= TARGET_UNK_EID + N_SJ_ROC)
sj_no_lead_chosen = rng.choice(sj_idx, size=N_SJ_NO_LEAD, replace=False)
df_work.loc[sj_no_lead_chosen, 'WOLeadName'] = np.nan
df_work.loc[sj_no_lead_chosen, 'FT_Owner']   = 'NOC_TEAM'

# Step 5: restore ROC account lead on N_SJ_ROC rows (subset of blanked)
# → net NaN = N_SJ_NO_LEAD - N_SJ_ROC = TARGET_UNK_EID = 825 (57.6%) ✅
sj_roc_chosen = rng.choice(sj_no_lead_chosen, size=N_SJ_ROC, replace=False)
df_work.loc[sj_roc_chosen, 'WOLeadName'] = 'ROC_ACCOUNT_LEAD'

print(f"  Part E — SJ anomaly: {n_sj_actual} rows "
      f"(UNKNOWN={N_SJ_UNKNOWN}, no_lead_net={N_SJ_NO_LEAD - N_SJ_ROC}, ROC={N_SJ_ROC})")

# ─────────────────────────────────────────────────────────────────────────────
# ASSEMBLE, SHUFFLE
# ─────────────────────────────────────────────────────────────────────────────

dirty_df = pd.concat(dirty_chunks, ignore_index=True)
for col in df_work.columns:
    if col not in dirty_df.columns:
        dirty_df[col] = np.nan
dirty_df = dirty_df[df_work.columns]

final_df = pd.concat([df_work, dirty_df], ignore_index=True)
final_df = final_df.iloc[rng.permutation(len(final_df))].reset_index(drop=True)

print("✅ Raw dirty dataset ready — pipeline will clean to 36,907")

Loaded clean base: 36,907 rows
  Part A — droppable rows: 1412 ✅
  Part B — timestamp corruption: 1327 rows (Zone 1: 193) ✅
  Part C — RFO noise: 5,321 rows (15%)
  Part D — missing NEType: 629 rows
  Part E — SJ anomaly: 1432 rows (UNKNOWN=912, no_lead_net=825, ROC=145)
✅ Raw dirty dataset ready — pipeline will clean to 36,907


## 6. Validate Distributions

In [8]:
print("\n── Distribution Validation ────────────────────────────────────────────")

def chk(label, val, target, tol=0):
    ok = abs(val - target) <= tol
    print(f"  {label:38s} {val:>6}  target={target}  {'✅' if ok else '❌'}")

chk("Total rows",                  len(final_df),                                          38319)
chk("Priority 4",                  int((final_df['Priority'] == 4).sum()),                  700)
chk("Blank Priority",              int(final_df['Priority'].isna().sum()),                    80)
chk("CANCELLED",                   int((final_df['FT Status'] == 'CANCELLED').sum()),        280)
chk("INPROG",                      int((final_df['FT Status'] == 'INPROG').sum()),           150)
chk("INVALID Fault Type",          int((final_df['Fault Type'] == 'INVALID').sum()),         100)
chk("Duplicate TICKETIDs",         int(final_df['TICKETID'].duplicated().sum()),               62)
chk("Corrupt timestamps",          int((final_df['StartDateTime'] == '2023-03-17 09:00:00').sum()), 1327)
chk("NEType blank",                int(final_df['NEType'].isna().sum()),                     629)

sj          = final_df[final_df['CITY'] == 'SAN JUAN CITY']
sj_unk_n    = int((sj['RFODescription'] == 'UNKNOWN-Under Investigation').sum())
sj_lead_n   = int(sj['WOLeadName'].isna().sum())
chk("SJ total rows",               len(sj),         n_sj_actual)
chk("SJ UNKNOWN RFO (exact)",      sj_unk_n,        N_SJ_UNKNOWN)
chk("SJ no WOLeadName (post-ROC)", sj_lead_n,       TARGET_UNK_EID)

non_can    = ~final_df['RFODescription'].isin(EXPECTED_RFO_VALUES)
z1_corrupt = int((final_df[final_df['ZONE'] == 'ZONE 1']['StartDateTime']
                  == '2023-03-17 09:00:00').sum())

print(f"\n  SJ UNKNOWN RFO%:      {sj_unk_n / len(sj) * 100:.1f}%  (target 63.7%)")
print(f"  SJ no WOLeadName%:    {sj_lead_n / len(sj) * 100:.1f}%  (net after ROC restore)")
print(f"  SJ ROC account rows:  {int((sj['WOLeadName'] == 'ROC_ACCOUNT_LEAD').sum())}  (target {N_SJ_ROC})")
print(f"  Zone 1 corrupted:     {z1_corrupt}  (target 193)")
print(f"  Non-canonical RFO:    {non_can.sum():,} ({non_can.mean() * 100:.1f}%)")

# ── Derive RFO_DIST from ZONE_RFO_COUNTS (single source of truth) ────────────
from collections import Counter
rfo_totals = Counter()
for zone_rfos in ZONE_RFO_COUNTS.values():
    for rfo, cnt in zone_rfos.items():
        rfo_totals[rfo] += cnt
total_rfo = sum(rfo_totals.values())
RFO_DIST = {rfo: cnt / total_rfo for rfo, cnt in rfo_totals.items()}

# ── RFO distribution (actual vs target, using clean base before dirty injection)
rfo_actual = df_work['Standardized RFO'].value_counts(normalize=True).round(3)
print("\nRFO distribution (actual clean base vs zone-count target):")
for rfo, target in sorted(RFO_DIST.items(), key=lambda x: -x[1]):
    actual = rfo_actual.get(rfo, 0)
    flag   = '⚠' if abs(actual - target) > 0.02 else '✓'
    print(f"  {flag} {rfo[:45]:<45} actual={actual:.3f}  target={target:.3f}")

# ── City coverage per zone ────────────────────────────────────────────────────
print("\nCities per zone (from site database):")
for zone in sorted(ZONE_CITIES.keys()):
    cities = sorted(c for c, _ in ZONE_CITIES[zone])
    print(f"  {zone}: {', '.join(cities)}")

# ── Volume and path summary ───────────────────────────────────────────────────
print(f"\nTotal raw tickets : {len(final_df):,}  (target 38,319)")
print(f"Missing NEType    : {final_df['NEType'].isna().sum():,}")
print(f"Null FT_Owner     : {final_df['FT_Owner'].isna().sum():,}  "
      f"(auto-resolved = {final_df['FT_Owner'].isna().mean()*100:.1f}%)")

print("\n✅ Section 6 complete — distributions validated")



── Distribution Validation ────────────────────────────────────────────
  Total rows                              38319  target=38319  ✅
  Priority 4                                700  target=700  ✅
  Blank Priority                             80  target=80  ✅
  CANCELLED                                 280  target=280  ✅
  INPROG                                    150  target=150  ✅
  INVALID Fault Type                        100  target=100  ✅
  Duplicate TICKETIDs                        62  target=62  ✅
  Corrupt timestamps                       1327  target=1327  ✅
  NEType blank                              629  target=629  ✅
  SJ total rows                            1432  target=1432  ✅
  SJ UNKNOWN RFO (exact)                    912  target=912  ✅
  SJ no WOLeadName (post-ROC)               825  target=825  ✅

  SJ UNKNOWN RFO%:      63.7%  (target 63.7%)
  SJ no WOLeadName%:    57.6%  (net after ROC restore)
  SJ ROC account rows:  145  (target 145)
  Zone 1 corrupted:     1

## 7. Save Raw File

In [9]:
final_df.to_csv(RAW_OUT, sep='\t', index=False, encoding='utf-8')

print(f"✅ Section 7 complete — raw file saved")
print(f"   Path    : {RAW_OUT}")
print(f"   Size    : {os.path.getsize(RAW_OUT) / 1024:.1f} KB")
print(f"   Rows    : {len(final_df):,}")
print(f"   Columns : {list(final_df.columns)}")


✅ Section 7 complete — raw file saved
   Path    : data/raw/fault_tickets/synthetic_dataset.csv
   Size    : 13299.4 KB
   Rows    : 38,319
   Columns : ['TICKETID', 'Region', 'ZONE', 'CITY', 'Area', 'FT Status', 'Priority', 'Urgency', 'OUTAGEDURATION', 'RESOLUTION_TIME_HOURS', 'Standardized RFO', 'RFODescription', 'RootCause', 'Fault Type', 'StartDateTime', 'ActionTaken', 'NEType', 'SLA_Compliant', 'FT_Owner', 'WOLeadName', 'WOOwnerGroup', 'FIELD_TIME_HOURS', 'DISPATCH_DELAY_HOURS', 'Priority_Urgency', 'REPORTDATE', 'ContactGroup', 'DESCRIPTION']


## 8. Smoke Test — Run Full Pipeline

In [10]:
from loading import load_data, load_site_database
from src.fault_ticket.pipeline import clean_data
from src.fault_ticket.metrics  import calculate_zone_summary
from config import ZONE_ORDER

print("\n── Smoke Test: Full Pipeline Run ──────────────────────────────────────")

# Load raw
df_loaded = load_data(RAW_OUT)
print(f"   Loaded  : {len(df_loaded):,} rows from synthetic file")

# Run pipeline (NCR / Region 3 scope)
df_clean = clean_data(df_loaded, region_scope='Region 3', save_output=False)
print(f"   Cleaned : {len(df_clean):,} rows retained")

# Zone summary
df_zone   = df_clean[df_clean['ZONE'].isin(ZONE_ORDER)].copy()
summary   = calculate_zone_summary(df_zone)

print("\n── Zone Summary (synthetic vs snapshot targets) ───────────────────────")
print(f"{'Zone':<8} {'Tickets':>8} {'SLA%':>7} {'MTTR_h':>8} {'FD%':>7}")
print("─" * 45)

SNAPSHOT_TARGETS = {
    'ZONE 1': dict(tickets=4306,  sla=83.1, mttr=62.6, fd=64.6),
    'ZONE 2': dict(tickets=7307,  sla=83.9, mttr=51.0, fd=62.8),
    'ZONE 3': dict(tickets=4393,  sla=80.6, mttr=32.3, fd=86.1),
    'ZONE 4': dict(tickets=7952,  sla=84.5, mttr=47.4, fd=56.2),
    'ZONE 5': dict(tickets=9131,  sla=79.6, mttr=77.8, fd=53.2),
    'ZONE 6': dict(tickets=3818,  sla=80.9, mttr=70.0, fd=61.4),
}

all_pass = True
for _, row in summary.iterrows():
    z    = row['ZONE']
    tgt  = SNAPSHOT_TARGETS.get(z, {})
    sla  = row['SLA_Compliance_Rate']
    mttr = row['MTTR']
    tix  = row['Ticket_Count']
    fd   = (df_zone[df_zone['ZONE']==z]['Resolution_Path']
            == 'Field_Dispatch_Restored').mean() * 100

    sla_ok  = abs(sla  - tgt.get('sla',  sla))  <= 5.0
    mttr_ok = abs(mttr - tgt.get('mttr', mttr)) <= 15.0
    fd_ok   = abs(fd   - tgt.get('fd',   fd))   <= 5.0
    ok      = sla_ok and mttr_ok and fd_ok
    all_pass = all_pass and ok
    flag    = '✓' if ok else '⚠'

    print(f"{flag} {z:<7} {tix:>8,} {sla:>7.1f}% {mttr:>7.1f}h {fd:>6.1f}%"
          f"  (tgt sla={tgt.get('sla','?')}% mttr={tgt.get('mttr','?')}h fd={tgt.get('fd','?')}%)")

# City coverage check
city_check = df_zone.groupby(['ZONE','CITY']).size().reset_index(name='Tickets')
print(f"\nCity coverage: {len(city_check)} zone-city combinations")
for z in ZONE_ORDER:
    cities = sorted(city_check[city_check['ZONE']==z]['CITY'].tolist())
    print(f"  {z}: {', '.join(cities) if cities else 'NONE ⚠'}")

print(f"\n{'✅ SMOKE TEST PASSED' if all_pass else '⚠  SMOKE TEST — check flagged zones'}")
print(f"   Pipeline retained {len(df_clean)/len(df_loaded)*100:.1f}% of raw tickets")
print(f"   Unique cities in cleaned output: {df_zone['CITY'].nunique()}")

INFO:pipeline:Pre-clean quality: 38,319 rows | score=99.1 | dupes=62 | blank_priority=80
INFO:pipeline:Saved 1000 rows → snapshot_pre_cleaning.csv



── Smoke Test: Full Pipeline Run ──────────────────────────────────────
   Loaded  : 38,319 rows from synthetic file


INFO:pipeline:
────────────────────────────────────────────────────────────
  PHASE 1 – STRUCTURAL FILTERING
────────────────────────────────────────────────────────────
INFO:pipeline:Step 1 – drop duplicate TICKETID: dropped 62 rows (remaining: 38,257)
INFO:pipeline:Step 2 – drop blank Priority: dropped 80 rows (remaining: 38,177)
INFO:pipeline:Step 3 – keep Priority 1-3: dropped 700 rows (remaining: 37,477)
INFO:pipeline:Step 4 – valid Urgency: dropped 25 rows (remaining: 37,452)
INFO:pipeline:Step 5 – valid FT Status: dropped 430 rows (remaining: 37,022)
INFO:pipeline:Step 6 – drop Fault Type INVALID: dropped 100 rows (remaining: 36,922)
INFO:pipeline:Saved 1000 rows → snapshot_after_phase1.csv
INFO:pipeline:
────────────────────────────────────────────────────────────
  PHASE 2 – DATA REPAIR & FIELD RECONSTRUCTION
────────────────────────────────────────────────────────────
INFO:pipeline:Step 7 – 629 missing NEType rows
INFO:pipeline:Saved 629 rows → missing_ne_type_tickets.csv
INF

   Cleaned : 36,922 rows retained

── Zone Summary (synthetic vs snapshot targets) ───────────────────────
Zone      Tickets    SLA%   MTTR_h     FD%
─────────────────────────────────────────────
✓ ZONE 5     9,134    79.6%    77.8h   53.2%  (tgt sla=79.6% mttr=77.8h fd=53.2%)
✓ ZONE 4     7,955    84.5%    47.4h   53.9%  (tgt sla=84.5% mttr=47.4h fd=56.2%)
✓ ZONE 2     7,311    83.9%    51.0h   62.8%  (tgt sla=83.9% mttr=51.0h fd=62.8%)
✓ ZONE 3     4,395    80.6%    32.3h   86.1%  (tgt sla=80.6% mttr=32.3h fd=86.1%)
✓ ZONE 1     4,307    83.1%    62.6h   64.6%  (tgt sla=83.1% mttr=62.6h fd=64.6%)
✓ ZONE 6     3,820    80.9%    70.0h   61.4%  (tgt sla=80.9% mttr=70.0h fd=61.4%)

City coverage: 18 zone-city combinations
  ZONE 1: CALOOCAN CITY, NAVOTAS, VALENZUELA CITY
  ZONE 2: QUEZON CITY
  ZONE 3: MALABON CITY, MARIKINA CITY, RIZAL
  ZONE 4: MANILA, PASIG CITY, SAN JUAN CITY
  ZONE 5: MAKATI CITY, PASAY CITY, PATEROS, TAGUIG CITY
  ZONE 6: BACOOR, LAS PINAS CITY, MUNTINLUPA CITY, PA

## 9. Output Summary

In [11]:
summary_table = pd.DataFrame({
    'File': [
        'synthetic_dataset.csv',
        'site_database_synthetic.csv',
        'cleaned_fault_ticket.csv',
    ],
    'Location': [
        'data/raw/fault_tickets/',
        'data/external/',
        'output/',
    ],
    'Description': [
        'Raw synthetic data — tab-delimited, pipeline-ready',
        'PLAID → Zone/City mapping for Region 3',
        'Pipeline output — run Section 8 to generate',
    ],
})

print("\n── Output Files ───────────────────────────────────────────────────────")
print(summary_table.to_string(index=False))

print("""
── To use in main pipeline notebooks ──────────────────────────────────────

    from loading import load_data
    from src.fault_ticket.pipeline import clean_data

    df       = load_data('data/raw/fault_tickets/synthetic_dataset.csv')
    df_clean = clean_data(df, region_scope='Region 3')
""")

dirty_table = pd.DataFrame({
    'Pattern': [
        'Duplicate TICKETID',
        'Blank Priority',
        'Priority = 4',
        'Invalid Urgency (99)',
        'CANCELLED / INPROG status',
        'Fault Type = INVALID',
        'Missing NEType',
        'Dirty / null RFODescription',
        'Blank RootCause',
        'Area code not in site DB',
    ],
    'Rate': [
        '~0.8%', '~1.5%', '~1.0%', '~2.0%', '~3.0%',
        '~5.0%', '~4.0%', '~30%',  '~20%',  '~2.0%',
    ],
    'Pipeline step that handles it': [
        'Step 1  — drop duplicate TICKETID',
        'Step 2  — drop blank Priority',
        'Step 3  — keep Priority 1–3 only',
        'Step 4  — valid Urgency 0–3',
        'Step 5  — valid FT Status',
        'Step 6  — drop Fault Type INVALID',
        'Step 7  — TF-IDF NEType inference',
        'Steps 10–15 — RFO standardisation cascade',
        'Step 10 — RootCause repair / fallback',
        'Step 16b — dropped as Unknown Area',
    ],
})

print("── Dirty-Data Patterns Injected ───────────────────────────────────────")
print(dirty_table.to_string(index=False))


── Output Files ───────────────────────────────────────────────────────
                       File                Location                                        Description
      synthetic_dataset.csv data/raw/fault_tickets/ Raw synthetic data — tab-delimited, pipeline-ready
site_database_synthetic.csv          data/external/             PLAID → Zone/City mapping for Region 3
   cleaned_fault_ticket.csv                 output/        Pipeline output — run Section 8 to generate

── To use in main pipeline notebooks ──────────────────────────────────────

    from loading import load_data
    from src.fault_ticket.pipeline import clean_data

    df       = load_data('data/raw/fault_tickets/synthetic_dataset.csv')
    df_clean = clean_data(df, region_scope='Region 3')

── Dirty-Data Patterns Injected ───────────────────────────────────────
                    Pattern  Rate             Pipeline step that handles it
         Duplicate TICKETID ~0.8%         Step 1  — drop duplicate TICKE